# Transfer Learning
This notebook is mainly to use pre-trained CNN model to complete the image classification task.

# 1. IMPORTS AND SETUP

In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns
import pandas as pd
import random

# Set random seeds for reproducibility
def set_seed(seed=21):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(21)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.9.0+cu126


# 2. DATA CONFIGURATION

In [ ]:
TRAIN_PATH = '/content/drive/MyDrive/CA Notebook/image/train'
TEST_PATH = '/content/drive/MyDrive/CA Notebook/image/test'

# Check if data exists
print("Checking data paths...")
print(f"Training path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Number of training files: {len([f for f in os.listdir(TRAIN_PATH) if f.lower().endswith(('.jpg', '.jpeg'))])}")
print(f"Number of test files: {len([f for f in os.listdir(TEST_PATH) if f.lower().endswith(('.jpg', '.jpeg'))])}")

Checking data paths...
Training path: /content/drive/MyDrive/CA Notebook/image/train
Test path: /content/drive/MyDrive/CA Notebook/image/test
Number of training files: 288
Number of test files: 60


 # 3. CUSTOM DATASET CLASS

In [ ]:
class FruitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = sorted(
            [f for f in os.listdir(root_dir) if f.lower().endswith(('.jpg', '.jpeg'))]
        )
        self.class_map = {"apple": 0, "banana": 1, "orange": 2, "mixed": 3}
        self.class_counts = {class_name: 0 for class_name in self.class_map.keys()}

        for filename in self.image_files:
            fn_lower = filename.lower()
            for cls_name in self.class_map.keys():
                if fn_lower.startswith(cls_name):
                    self.class_counts[cls_name] += 1
                    break

        print(f"Dataset class distribution: {self.class_counts}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.root_dir, filename)

        with Image.open(image_path) as img:
            if img.mode in ('P', 'RGBA', 'LA', 'PA'):
                rgba = img.convert('RGBA')
                background = Image.new('RGB', rgba.size, (255, 255, 255))
                if rgba.mode == 'RGBA':
                    background.paste(rgba, mask=rgba.split()[3])
                else:
                    background.paste(rgba)
                image = background
            else:
                image = img.convert('RGB')

        if self.transform:
            image = self.transform(image)

        # Label parsing
        label = -1
        fn_lower = filename.lower()
        for cls_name, cls_idx in self.class_map.items():
            if fn_lower.startswith(cls_name):
                label = cls_idx
                break

        if label == -1:
            raise ValueError(f"Cannot parse class from filename: {filename}")
        return image, label

    def get_class_weights(self):
        """Calculate class weights for handling class imbalance"""
        total = len(self.image_files)
        weights = []
        for cls_name in self.class_map.keys():
            count = self.class_counts[cls_name]
            if count == 0:
                weights.append(0)
            else:
                weights.append(total / (len(self.class_map) * count))
        return torch.tensor(weights, dtype=torch.float32)

#4. DATA PREPROCESSING AND AUGMENTATION

In [ ]:
image_size = 128

train_transform = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# 5. LOAD AND SPLIT DATASETS (ONCE)

In [ ]:
print("Loading datasets...")
full_train_dataset = FruitDataset(TRAIN_PATH, transform=train_transform)
test_dataset = FruitDataset(TEST_PATH, transform=test_transform)

print(f"Full training set size: {len(full_train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

class_names = ['apple', 'banana', 'orange', 'mixed']

Loading datasets...
Dataset class distribution: {'apple': 75, 'banana': 71, 'orange': 72, 'mixed': 70}
Dataset class distribution: {'apple': 19, 'banana': 18, 'orange': 18, 'mixed': 5}
Full training set size: 288
Test set size: 60


# 6. MODEL ARCHITECTURES

In [ ]:
class ResFruitNet(nn.Module):
    def __init__(self, num_classes=4):
        super(ResFruitNet, self).__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        num_filters = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_filters, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        if len(x.shape) == 2:
            x = x.view(x.size(0), -1, 1, 1)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

class EfficientFruitNet(nn.Module):
    def __init__(self, num_classes=4):
        super(EfficientFruitNet, self).__init__()
        self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        num_filters = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_filters, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        self._freeze_backbone_layers()

    def _freeze_backbone_layers(self):
        for name, param in self.backbone.features.named_parameters():
            if 'features.7' in name or 'features.8' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

class ConvNeXtFruitNet(nn.Module):
    def __init__(self, num_classes=4):
        super(ConvNeXtFruitNet, self).__init__()
        self.backbone = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
        num_filters = self.backbone.classifier[2].in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(num_filters, eps=1e-6),
            nn.Dropout(0.4),
            nn.Linear(num_filters, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        self._freeze_backbone_layers()

    def _freeze_backbone_layers(self):
        for name, param in self.backbone.features.named_parameters():
            if '7' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

# 7. TRAINING UTILITIES

In [ ]:
def create_dataloaders(full_train_dataset, test_dataset, batch_size=32):
    """Create fresh dataloaders with proper isolation"""
    set_seed(21)  # Reset seed for consistent splits

    # Split validation set
    train_size = int(0.8 * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size
    train_db, val_db = torch.utils.data.random_split(
        full_train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(21)
    )

    # Validation set uses test preprocessing
    val_db.dataset.transform = test_transform

    # Create data loaders
    train_loader = DataLoader(train_db, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_db, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader

def train_model(model_class, model_name, train_loader, val_loader, class_weights, num_epochs=40):
    """Train a model and return best weights and history"""

    # Clear GPU cache and reset random state
    torch.cuda.empty_cache()
    set_seed(21)

    # Create fresh model instance
    model = model_class(num_classes=4).to(device)

    # Loss function with class weights
    class_weights_device = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_device, label_smoothing=0.02)

    # Optimizer (only trainable parameters)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.0003,
        weight_decay=1e-2,
        betas=(0.9, 0.999)
    )

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    # Training history
    train_history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'learning_rate': []
    }

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    min_val_loss = float('inf')

    print(f"\n{'='*70}")
    print(f"Training {model_name}")
    print(f"{'='*70}")
    print(f"{'Epoch':^5} | {'Tr Loss':^8} | {'Tr Acc':^7} | {'Val Loss':^8} | {'Val Acc':^7} | {'LR':^8} | {'Save'}")
    print("-" * 85)

    for epoch in range(num_epochs):
        start_time = time.time()

        # Training phase
        model.train()
        tr_loss, tr_corr, tr_total = 0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            tr_loss += loss.item() * inputs.size(0)
            tr_corr += (torch.max(outputs, 1)[1] == labels).sum().item()
            tr_total += labels.size(0)

        # Validation phase
        model.eval()
        v_loss, v_corr, v_total = 0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                v_loss += loss.item() * inputs.size(0)
                v_corr += (torch.max(outputs, 1)[1] == labels).sum().item()
                v_total += labels.size(0)

        # Calculate metrics
        epoch_tr_loss = tr_loss / tr_total
        epoch_tr_acc = tr_corr / tr_total
        epoch_v_loss = v_loss / v_total
        epoch_v_acc = v_corr / v_total
        current_lr = optimizer.param_groups[0]['lr']

        # Update learning rate
        scheduler.step()

        # Save best model
        save_msg = ""
        if epoch_v_acc > best_val_acc:
            best_val_acc = epoch_v_acc
            min_val_loss = epoch_v_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), f'best_{model_name.lower()}_model.pth')
            save_msg = "⭐Acc"
        elif epoch_v_acc == best_val_acc and epoch_v_loss < min_val_loss:
            min_val_loss = epoch_v_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), f'best_{model_name.lower()}_model.pth')
            save_msg = "💎Loss"

        # Record history
        train_history['train_loss'].append(epoch_tr_loss)
        train_history['train_acc'].append(epoch_tr_acc)
        train_history['val_loss'].append(epoch_v_loss)
        train_history['val_acc'].append(epoch_v_acc)
        train_history['learning_rate'].append(current_lr)

        # Print progress
        epoch_time = time.time() - start_time
        print(f"{epoch+1:^5} | {epoch_tr_loss:^8.4f} | {epoch_tr_acc:^7.2%} | "
              f"{epoch_v_loss:^8.4f} | {epoch_v_acc:^7.2%} | {current_lr:^8.2e} | {save_msg}")

        # Show training curves at final epoch
        if epoch == num_epochs - 1:
            plot_training_curves(train_history, model_name)

    print(f"\nTraining completed! Best validation accuracy: {best_val_acc:.2%}")
    return best_model_wts, train_history, best_val_acc

def plot_training_curves(train_history, model_name):
    """Plot training curves"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss curves
    axes[0].plot(train_history['train_loss'], label='Training Loss')
    axes[0].plot(train_history['val_loss'], label='Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy curves
    axes[1].plot(train_history['train_acc'], label='Training Accuracy')
    axes[1].plot(train_history['val_acc'], label='Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Learning rate curve
    axes[2].plot(train_history['learning_rate'], label='Learning Rate')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Learning Rate Schedule')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f'{model_name} - Training Progress', fontsize=14)
    plt.tight_layout()
    plt.show()


# 8. TESTING UTILITIES

In [ ]:
def test_model(model_class, model_name, test_loader, class_names):
    """Test a model with TTA and post-processing"""

    # Load best model
    model = model_class(num_classes=4).to(device)
    model.load_state_dict(torch.load(f'best_{model_name.lower()}_model.pth'))
    model.eval()

    print(f"\n{'='*60}")
    print(f"{model_name} Test Results")
    print(f"{'='*60}")

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            batch_preds = []

            for i in range(inputs.size(0)):
                img = inputs[i]
                img_flip = torch.flip(img, dims=[2])
                batch = torch.stack([img, img_flip]).to(device)

                outputs = model(batch)
                probs = F.softmax(outputs, dim=1)
                avg_probs = (probs[0] * 0.55) + (probs[1] * 0.45)

                prob_val, pred_idx = torch.max(avg_probs, 0)
                pred_idx = pred_idx.item()

                # Post-processing: recover single fruits misclassified as Mixed
                if pred_idx == 3 and prob_val < 0.60:
                    _, top2 = torch.topk(avg_probs, 2)
                    second_choice = top2[1].item()
                    if second_choice in [0, 1, 2]:
                        pred_idx = second_choice

                batch_preds.append(pred_idx)
                all_probs.append(avg_probs.cpu().numpy())

            all_preds.extend(batch_preds)
            all_labels.extend(labels.numpy().tolist())

    # Calculate final accuracy
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    final_acc = np.mean(all_preds == all_labels)

    print(f"Test Accuracy: {final_acc:.2%}")
    print(f"Number of test samples: {len(all_labels)}")
    print()

    # Classification report
    print("Detailed Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.show()

    # Per-class statistics
    print("\nPer-Class Accuracy Statistics:")
    for i, class_name in enumerate(class_names):
        class_indices = np.where(all_labels == i)[0]
        if len(class_indices) > 0:
            class_acc = np.mean(all_preds[class_indices] == all_labels[class_indices])
            print(f"{class_name:10}: {class_acc:.2%} ({len(class_indices)} samples)")

    return final_acc, all_preds, all_labels


# 9. TRAIN AND EVALUATE ALL MODELS


In [ ]:
# Calculate class weights once
class_weights = full_train_dataset.get_class_weights()
print(f"Class weights: {class_weights}")

# Model configurations
model_configs = [
    (ResFruitNet, "ResFruitNet"),
    (EfficientFruitNet, "EfficientFruitNet"),
    (ConvNeXtFruitNet, "ConvNeXtFruitNet")
]

results = {}

for model_class, model_name in model_configs:
    # Clear GPU memory and reset random state
    torch.cuda.empty_cache()
    set_seed(21)

    # Create fresh dataloaders for each model
    train_loader, val_loader, test_loader = create_dataloaders(
        full_train_dataset, test_dataset, batch_size=32
    )

    # Train model
    best_weights, history, best_val_acc = train_model(
        model_class, model_name, train_loader, val_loader, class_weights
    )

    # Test model
    test_acc, preds, labels = test_model(
        model_class, model_name, test_loader, class_names
    )

    # Save results
    results[model_name] = {
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'preds': preds,
        'labels': labels
    }

    # Force garbage collection
    import gc
    gc.collect()
    torch.cuda.empty_cache()

# 10. MODEL COMPARISON


In [ ]:
print(f"\n{'='*70}")
print("MODEL COMPARISON SUMMARY")
print(f"{'='*70}")

print(f"\n{'Model':<20} {'Best Val Acc':<15} {'Test Acc':<15}")
print("-" * 50)

for model_name, result in results.items():
    print(f"{model_name:<20} {result['best_val_acc']:<15.2%} {result['test_acc']:<15.2%}")

# Plot comparison
models_list = list(results.keys())
val_accs = [results[m]['best_val_acc'] for m in models_list]
test_accs = [results[m]['test_acc'] for m in models_list]

x = np.arange(len(models_list))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, val_accs, width, label='Validation Accuracy', color='skyblue')
bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy', color='lightcoral')

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models_list)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1%}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Save comprehensive results to CSV
results_df = pd.DataFrame({
    'Model': models_list,
    'Best_Validation_Accuracy': val_accs,
    'Test_Accuracy': test_accs
})

results_df.to_csv('model_comparison_results.csv', index=False)
print(f"\nResults saved to 'model_comparison_results.csv'")